[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiptos/is_notes/blob/main/rl_cartpole.ipynb)

# Reinforcement Learning

Reinforcement learning problems involve learning what to do, namely how to map situations to actions, so as to maximize a numerical reward signal.

In this example, let's take a look at the CartPole example: a pole is attached to a cart, which moves left or right. The agent receives a reward of +1 for every step in which the pole stays upright, and the episode ends when the pole falls or the cart leaves the track (or after 500 steps). We train the policy with the REINFORCE algorithm (policy gradient).

## Install packages

In [ ]:
!pip install -q "gymnasium[classic-control]" moviepy

## Import libraries

In [ ]:
import gymnasium as gym
import numpy as np
from collections import deque
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from gymnasium.utils.save_video import save_video
from torch.distributions import Categorical
from IPython.display import Video, display

## Control options

In [ ]:
gamma=0.99
# render="human"  # shows a window (only locally, not in Colab)
render="rgb_array_list"  # keeps the frames of each episode, to save videos
log_interval=10
seed=543
max_episodes=2000  # stop if the problem is not solved before

env = gym.make('CartPole-v1', render_mode=render)
env.reset(seed=seed)
torch.manual_seed(seed)

## $\pi(s, a)$ neural network

In [ ]:
class Policy(nn.Module):
    def __init__(self):
        super(Policy, self).__init__()
        self.affine1 = nn.Linear(4, 128)
        self.dropout = nn.Dropout(p=0.6)
        self.affine2 = nn.Linear(128, 2)

        self.saved_log_probs = []
        self.rewards = []

    def forward(self, x):
        x = self.affine1(x)
        x = self.dropout(x)
        x = F.relu(x)
        action_scores = self.affine2(x)
        return F.softmax(action_scores, dim=1)

## Settings

In [ ]:
policy = Policy()
optimizer = optim.Adam(policy.parameters(), lr=1e-2)
eps = np.finfo(np.float32).eps.item()

## Functions

In [ ]:
def select_action(state):
    state = torch.from_numpy(state).float().unsqueeze(0)
    probs = policy(state)
    m = Categorical(probs)
    action = m.sample()
    policy.saved_log_probs.append(m.log_prob(action))
    return action.item()


# finish_episode() is called at the end of every episode to:
#	1.	Compute the returns (discounted sum of rewards)
#	2.	Normalize them (variance reduction)
#	3.	Compute the policy gradient loss
#	4.	Perform a backward pass and optimize the policy network
#	5.	Clear stored rewards and log-probs for the next episode
def finish_episode():
    R = 0
    policy_loss = []
    returns = deque()
    for r in policy.rewards[::-1]:
        R = r + gamma * R
        returns.appendleft(R)
    returns = torch.tensor(returns)
    returns = (returns - returns.mean()) / (returns.std() + eps)
    for log_prob, R in zip(policy.saved_log_probs, returns):
        policy_loss.append(-log_prob * R)
    optimizer.zero_grad()
    policy_loss = torch.cat(policy_loss).sum()
    policy_loss.backward()
    optimizer.step()
    del policy.rewards[:]
    del policy.saved_log_probs[:]

def save_episode(episode: int):
    return episode % 10 == 0


## Main cycle

In each episode, the agent plays with the current policy, sampling the actions from the probabilities given by the network. At the end of the episode, the policy is updated: the actions followed by high returns become more probable. The problem is considered solved when the average reward reaches the threshold of the environment (475). A video of every 10th episode is saved in the `videos` folder.

In [ ]:
running_reward = 10
step_starting_index = 0
for i_episode in range(1, max_episodes + 1):
    state, _ = env.reset()
    ep_reward = 0
    for t in range(1, 10000):  # Don't infinite loop while learning
        action = select_action(state)
        state, reward, terminated, truncated, _ = env.step(action)
        if render=="human":
            env.render()
        policy.rewards.append(reward)
        ep_reward += reward
        # the episode ends when the pole falls (terminated) or after 500 steps (truncated)
        if terminated or truncated:
            break

    if render=="rgb_array_list":
        save_video(env.render(),
               "videos", fps=int(env.metadata["render_fps"]), step_starting_index=step_starting_index,
               episode_index=i_episode, episode_trigger=save_episode)
    running_reward = 0.05 * ep_reward + (1 - 0.05) * running_reward
    finish_episode()
    if i_episode % log_interval == 0:
        print('Episode {}\tLast reward: {:.2f}\tAverage reward: {:.2f}'.format(
            i_episode, ep_reward, running_reward))
    if running_reward > env.spec.reward_threshold:
        print("Solved! Running reward is now {} and "
              "the last episode runs to {} time steps!".format(running_reward, t))
        break
else:
    print(f"Not solved after {max_episodes} episodes. Running reward: {running_reward:.2f}")

torch.save(policy.state_dict(), 'cartpole_policy.pth')
env.close()

# Autonomous play...

Let's load the trained policy and let it play, always choosing the most probable action. The last episode is recorded and shown below.

Locally (not in Colab), you can also play yourself, with the keys `a` and `d`, setting `USER_PLAY = True`.

In [ ]:
from gymnasium.utils import play

# Allows for loading and running a pre-trained model,
# or for the user to interact with the environment.

USER_PLAY = False
MODEL_FILE = "cartpole_policy.pth"

if USER_PLAY:
    env = gym.make("CartPole-v1", render_mode="rgb_array")
    key_action_map = {(ord('a'),): 0, (ord('d'),): 1}  # Map 'a' and 'd' to going left and right
    play.play(env, keys_to_action=key_action_map, fps=15)  # Lower fps cause its too hard otherwise
else:
    env = gym.make("CartPole-v1", render_mode="rgb_array_list")

    model = Policy()
    model.load_state_dict(torch.load(MODEL_FILE, weights_only=True))
    model.eval()  # disables the dropout
    n_episodes = 5

    for e in range(n_episodes):
        # 0: Cart position, 1: Cart Velocity, 2: Pole Angle, 3: Pole Angular Velocity
        state, _ = env.reset()
        done = False
        score = 0
        while not done:
            with torch.no_grad():
                probs = model(torch.from_numpy(state).float().unsqueeze(0))
            # 0: Left, 1: Right (the most probable action)
            action = probs.argmax(dim=1).item()
            state, reward, terminated, truncated, _ = env.step(action)
            score += reward
            done = terminated or truncated

        print("episode: {}, score: {}".format(e, score))

    # save and show the video of the last episode
    save_video(env.render(), "videos", fps=int(env.metadata["render_fps"]), name_prefix="cartpole-play",
               episode_trigger=lambda episode: True)
    env.close()
    display(Video("videos/cartpole-play-episode-0.mp4", embed=True))